# `points_membership_mask()`

The geometry function `nematics3d.geometry.points_membership_mask()` returns a boolean mask indicating whether each input point appears exactly in a candidate point set. It compares complete rows rather than individual coordinates and preserves the order of the first point collection.

The function is intended for exact point or lattice-index membership. Numerical dtypes may differ between the two inputs, but floating-point coordinates are still compared exactly rather than with a tolerance.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** Run the following cell to import `NumPy` and `Nematics3D`; no setup detail is needed to understand the examples.


In [ ]:
import numpy as np
import nematics3d as n3d

## Inputs and outputs

The public signature is:

```python
points_membership_mask(points, candidates)
```

### Accepted point representations

Both `points` and `candidates` must be finite real point collections with shape `(N, D)` and `(M, D)`. The coordinate dimension $D$ may be any positive dimension, but it must match between the two inputs. Empty collections are allowed.

The two arrays may use different numerical dtypes. `Nematics3D` promotes them to a common numerical representation before testing row equality, so equal coordinates such as integer `[1, 2]` and floating-point `[1.0, 2.0]` compare as the same point.

### Returned result

The function returns a one-dimensional boolean `NumPy` array with length equal to the number of rows in `points`. Entry `i` is `True` when `points[i]` occurs at least once in `candidates`, and `False` otherwise. Duplicate rows in either input do not change this interpretation.


## Examples


### Minimal example

Each row of `points` is tested against the complete rows of `candidates`.


In [ ]:
points = np.array([
    [0, 0],
    [1, 2],
    [3, 4],
])
candidates = np.array([
    [3, 4],
    [0, 0],
])

mask = n3d.geometry.points_membership_mask(points, candidates)
print(mask)
print(points[mask])

### Different numerical dtypes

Membership is numerical rather than a comparison of raw memory bytes. Therefore equal coordinates can match even when the two arrays use different numerical dtypes.


In [ ]:
points_i32 = np.array([[1, 2], [3, 4]], dtype=np.int32)
candidates_f64 = np.array([[1.0, 2.0]], dtype=np.float64)

print(n3d.geometry.points_membership_mask(points_i32, candidates_f64))

### Floating-point comparison is exact

This function does not apply a geometric tolerance. Two floating-point points match only when all corresponding coordinates are numerically equal after dtype promotion. Use a nearest-point or tolerance-based geometric query instead when approximate equality is intended.


In [ ]:
points_float = np.array([
    [1.0, 2.0],
    [1.0, 2.0 + 1e-12],
])
candidates_float = np.array([[1.0, 2.0]])

print(n3d.geometry.points_membership_mask(points_float, candidates_float))

### Empty collections

An empty `points` array produces an empty mask. An empty `candidates` array produces an all-`False` mask with one entry per input point.


In [ ]:
print(n3d.geometry.points_membership_mask(np.empty((0, 2)), candidates))
print(n3d.geometry.points_membership_mask(points, np.empty((0, 2))))

## Details

After validating the two point collections, the function promotes both arrays to a common numerical dtype and stores them contiguously. Each coordinate column is then represented as one field of a structured `NumPy` dtype, so each point can be treated as one record during `np.isin()`.

This preserves vectorized row-wise membership without constructing an `(N, M, D)` pairwise comparison array. It also avoids the raw-byte comparison problem of viewing rows as `np.void`: numerical values such as `+0.0` and `-0.0` follow ordinary numerical equality semantics.

For typical point collections, the result therefore has the semantics of exact set membership while retaining one boolean output for every row of the original `points` array.


## Used by

`QPlane._helper_detect_defect()` uses `points_membership_mask()` to mark which integer plane-grid indices belong to the defect-vicinity index set. The returned mask becomes the basis for identifying sampled directors near detected defects.
